### Install Libraries

In [ ]:
!pip install datasets
!pip install youtube-comment-downloader
!pip install youtube-search-python
!pip install "httpx<0.24.0"
!pip install psaw
!pip install requests beautifulsoup4

### Scrape Youtube comment data

In [ ]:
import os
import time
from youtubesearchpython import VideosSearch
from youtube_comment_downloader import YoutubeCommentDownloader

"""
domains = {
    "gaming": ["gaming deutsch", "spiel bewertung", "pc spiele test", "konsole vergleich deutsch"],
    "music": ["deutschrap 2024", "musikvideo reaktion", "musik bewertung"],
    "products": ["produkttest deutsch", "smartphone review deutsch", "technik empfehlung"],
    "politics": ["nachrichten meinung deutsch", "politische debatte deutschland", "bundestag diskussion"],
    "news": ["tagesschau aktuell", "zdf heute meinung", "nachrichten kommentar"],
    "travel": ["hotel erfahrung deutsch", "reisebericht deutschland", "urlaub bewertung"]
}
"""

domains = {
    "gaming": [
        "bester shooter 2024 deutsch",
        "story game empfehlung deutsch",
        "spiel ist enttäuschung deutsch",
        "gameplay kritik deutsch",
        "open world spiele meinung deutsch"
    ],
    "music": [
        "deutscher rap diss track",
        "musikvideo kritik deutsch",
        "emotionales lied deutsch",
        "kommentare zu deutschem song",
        "reaktion auf musik deutsch"
    ],
    "products": [
        "billig vs teuer test deutsch",
        "produkt enttäuscht mich deutsch",
        "top 5 gadgets 2024 deutsch",
        "unboxing erfahrung deutsch",
        "schlechtester kauf deutsch"
    ],
    "politics": [
        "deutschland politik meinung 2024",
        "diskussion migration deutschland",
        "klimapolitik kritik deutsch",
        "regierung versagt meinung",
        "afd vs grüne debatte"
    ],
    "news": [
        "kommentar zu aktuellen nachrichten",
        "reaktion auf tagesschau",
        "meinung zu wirtschaftslage deutsch",
        "meinung zum wahlkampf deutschland",
        "deutschland krise meinung"
    ],
    "travel": [
        "airbnb erfahrung deutsch",
        "hotel enttäuschung urlaub",
        "reise nach spanien deutsch vlog",
        "flugverspätung erfahrung deutsch",
        "schlechtester urlaub meines lebens"
    ]
}

videos_per_query = 120
max_comments_per_video = 1000
output_file = "youtube_comments_clean.txt"
processed_log = "processed_videos.log"
target_size_MB = 40  # Stop after 40MB

# Load URLs
if os.path.exists(processed_log):
    with open(processed_log, "r", encoding="utf-8") as f:
        processed = set(line.strip() for line in f)
else:
    processed = set()

downloader = YoutubeCommentDownloader()

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

with open(output_file, "a", encoding="utf-8") as out_file, open(processed_log, "a", encoding="utf-8") as log_file:
    for domain, queries in domains.items():
        for query in queries:
            print(f"\nSearching: {query}")
            try:
                videos_search = VideosSearch(query, limit=videos_per_query)
                results = videos_search.result()["result"]
            except Exception as e:
                print(f"Search failed: {e}")
                continue

            for video in results:
                url = video["link"]
                if url in processed:
                    print(f"Skipping: already processed {url}")
                    continue

                print(f"Processing: {video['title']}\n    URL: {url}")
                try:
                    count = 0
                    for comment in downloader.get_comments_from_url(url, sort_by=0):
                        text = comment.get("text", "").strip()
                        if text:
                            clean_text = text.replace("\r", " ").replace("\n", " ").strip()
                            out_file.write(clean_text + "\n")
                            count += 1

                            if count >= max_comments_per_video:
                                break
                            if file_size_mb(output_file) >= target_size_MB:
                                print(f"\nReached {target_size_MB}MB — stopping.")
                                raise StopIteration

                    print(f"   Saved {count} comments.")
                    log_file.write(url + "\n")
                    log_file.flush()

                except StopIteration:
                    raise
                except Exception as e:
                    print(f"Comment scraping error: {e}")
                    continue

                time.sleep(1)

### Download Amazon review data

In [ ]:
import csv
import pandas as pd

df = pd.read_csv(
    "amazon_review.csv",
    encoding="latin1",
    sep="\t",
    on_bad_lines="skip",
    engine="python"
)

### Scrape Reddit comment data

In [ ]:
import os
import time
from psaw import PushshiftAPI
import datetime
import re

domains = {
    "gaming": [
        "open world spiele meinung deutsch", "story game enttäuschung", "shooter zu schwer", "was zockt ihr gerade", "game release meinung"
    ],
    "music": [
        "deutscher rap meinung", "track ist überbewertet", "was hört ihr gerade", "lieblingsalbum deutsch", "musikalischer fail"
    ],
    "products": [
        "mein schlechtester kauf", "technik enttäuscht mich", "smartphone meinung 2024", "amazon bewertung deutsch", "top gadgets deutsch"
    ],
    "politics": [
        "meinung zu migration deutschland", "klimapolitik kritik", "afd meinung", "regierung 2024", "deutschland wahl"
    ],
    "news": [
        "meinung zu aktuellen nachrichten", "deutschland wirtschaftslage", "nachrichten kommentar", "tagesschau meinung", "panorama debatte"
    ],
    "travel": [
        "reise war enttäuschung", "hotel horror erfahrung", "urlaubspanne 2024", "airbnb meinung", "reise gebucht bereue es"
    ]
}

comments_per_query = 300
output_file = "reddit_comments_clean.txt"
processed_log = "processed_reddit_queries.log"
target_size_MB = 40

api = PushshiftAPI()

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

if os.path.exists(processed_log):
    with open(processed_log, "r", encoding="utf-8") as f:
        processed_queries = set(f.read().splitlines())
else:
    processed_queries = set()

with open(output_file, "a", encoding="utf-8") as out_file, open(processed_log, "a", encoding="utf-8") as log_file:
    for domain, queries in domains.items():
        for query in queries:
            if query in processed_queries:
                print(f"Skipping (already processed): {query}")
                continue

            print(f"\nSearching Reddit for: {query}")
            try:
                gen = api.search_comments(q=query, lang='de', limit=comments_per_query,
                                          after=int(datetime.datetime(2022, 1, 1).timestamp()))

                count = 0
                for comment in gen:
                    text = comment.body.strip()
                    if not text or text in ["[deleted]", "[removed]"]:
                        continue
                    clean = re.sub(r"\s+", " ", text)
                    out_file.write(clean + "\n")
                    count += 1

                    if file_size_mb(output_file) >= target_size_MB:
                        print(f"\nReached {target_size_MB}MB — stopping.")
                        raise StopIteration

                print(f"Saved {count} comments for: {query}")
                log_file.write(query + "\n")
                log_file.flush()

            except StopIteration:
                raise
            except Exception as e:
                print(f"Error while fetching: {e}")
                continue

            time.sleep(1)


### Scrape Holidaycheck review data

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import os

# List of hotel page URLs
hotel_urls = [
    "https://www.holidaycheck.de/hi/hotel-am-steinplatz/8b8cd9f5-6c95-4b9d-b3b3-25b08a3fd5c1",
]

output_file = "holidaycheck_comments_clean.txt"
target_size_MB = 40

headers = {
    "User-Agent": "Mozilla/5.0"
}

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0

with open(output_file, "a", encoding="utf-8") as f_out:
    for base_url in hotel_urls:
        print(f"\nScraping reviews from: {base_url}")
        page = 1

        while True:
            full_url = f"{base_url}/reviews?page={page}"
            try:
                res = requests.get(full_url, headers=headers, timeout=10)
                if res.status_code != 200:
                    print(f"Failed to fetch: {full_url}")
                    break

                soup = BeautifulSoup(res.text, "html.parser")
                review_divs = soup.select("div.review-text")

                if not review_divs:
                    print("No more reviews found.")
                    break

                for div in review_divs:
                    text = div.get_text(separator=" ", strip=True)
                    if len(text.split()) < 5:
                        continue
                    f_out.write(text + "\n")

                    if file_size_mb(output_file) >= target_size_MB:
                        print(f"Reached {target_size_MB}MB — stopping.")
                        raise StopIteration

                print(f"Page {page} done")
                page += 1
                time.sleep(1)

            except StopIteration:
                break
            except Exception as e:
                print(f"⚠️ Error: {e}")
                break

### Process holidaycheck dataset

In [ ]:
import pandas as pd

# Read tab-separated file with unknown number of fields
df = pd.read_csv("holidaycheck.clean.filtered.tsv", sep="\t", header=None, quoting=3, on_bad_lines='skip')

# View first few lines
# print(df.head())

# df = pd.read_csv("holidaycheck.clean.filtered.tsv")
df = df.sample(frac=1, random_state=42)
df.to_csv("holidaycheck_reviews_shuffled.csv", index=False)

In [ ]:
df = pd.read_csv("holidaycheck_reviews_shuffled.csv", sep="\t", header=None, quoting=3, on_bad_lines='skip')
print(df.head(50))

                                                    0
0                                                 0,1
1   5,Die Freundlichkeit ist von Person zur Person...
2   5,"Ruhe und Erholung gesucht und definitiv gef...
3   6,"Sehr schönes, modernes und großes  Hotel Da...
4   2,"Die Animation war unerträglich albern und h...
5   6,Fahrt vom Flughafen ca.45 Min. Umgebung sehr...
6   3,"10 Minuten bis zum Hotel, Mit dem Taxi konn...
7   5,"Entspannter Urlaub im Familienhotel Das Hot...
8   1,"Reinfall ! Diese gesamte Anlage ist ein abs...
9   6,"Das Frühstück war ein Traum wie es sich jed...
10  6,"Toller Urlaub Wir sind als zwei Freundinnen...
11  6,"Es gibt in der Anlage zwei Poollandschaften...
12  6,"Empfehle das Hotel weiter. Super Urlaub. Wi...
13  5,"Ein Traum aus 1001 Nacht! Die Lobby bereits...
14        6,"Absolut Top, sehr freundliches Personal"
15                                   6,Schöner Strand
16                                      6,Alles super
17  6,"Super zufrieden Das h

In [ ]:
import pandas as pd
df = pd.read_csv("holidaycheck_reviews_shuffled.csv", dtype={"stars": str}, low_memory=False)
df['stars'] = df['stars'].str.strip().astype(int)
df['stars'] = df['stars'].apply(lambda x: 'positive' if x >= 4 else 'negative')

print(df.head())


      stars                                        review_text
0  positive  Die Freundlichkeit ist von Person zur Person v...
1  positive  Ruhe und Erholung gesucht und definitiv gefund...
2  positive  Sehr schönes, modernes und großes  Hotel Das H...
3  negative  Die Animation war unerträglich albern und hatt...
4  positive  Fahrt vom Flughafen ca.45 Min. Umgebung sehr s...


In [ ]:
df.to_csv("holidaycheck_reviews_binary.csv", index=False)

In [ ]:
df = pd.read_csv("holidaycheck_reviews_binary.csv")
print(df['sentiment'].value_counts())

sentiment
positive    3135591
negative     835869
Name: count, dtype: int64


In [ ]:
positive_df = df[df['sentiment'] == 'positive']
negative_df = df[df['sentiment'] == 'negative']

# Downsample the positive class to match the number of negatives
positive_sampled = positive_df.sample(n=len(negative_df), random_state=42)

# Concatenate and shuffle
balanced_df = pd.concat([positive_sampled, negative_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(balanced_df['sentiment'].value_counts())

sentiment
positive    835869
negative    835869
Name: count, dtype: int64


In [ ]:
balanced_df.to_csv("holidaycheck_reviews_binary_balanced.csv", index=False)

### Clean SB10k dataset

In [ ]:
import pandas as pd

df = pd.read_csv("sb10k.tsv", sep="\t", usecols=["Sentiment", "Text"])
df.to_csv("sb10k_cleaned.csv", index=False)

### Clean GermEval2017 dataset

In [ ]:
import pandas as pd

df = pd.read_csv(
    "germeval_train.tsv",
    sep="\t",
    header=None,
    names=["ID", "Text", "Relevance", "Sentiment", "AspectPolarity"],
    usecols=["Text", "Sentiment"]
)

# Reorder columns
df = df[["Sentiment", "Text"]]
df.to_csv("germeval2017_cleaned.csv", index=False)